# Skynode: train your own detector in Google Colab

Everything here runs on Google's computers. Your laptop stores and runs nothing.

**Before you start**
1. Film planes and drones with your phone's normal camera app (steady, sky in view, a few different days and places help a lot).
2. Upload the videos to Google Drive in a folder called **`skynode/videos`** (My Drive ▸ skynode ▸ videos).
   Name a video like **`drone_garden.mp4`** or **`airplane_morning.mp4`** when it only shows that one thing. The first word before the `_` tells the notebook which box to keep. Use another word (e.g. `sky_clear.mp4`) for mixed or empty-sky clips. Empty sky is useful too: it teaches the model what *not* to flag.
3. **Runtime ▸ Change runtime type ▸ T4 GPU.**
4. Run the cells top to bottom. Change settings only in the **Settings** cell.

If Colab disconnects, just run everything again. Training picks up where it stopped, because progress is saved to your Drive.

## 1 · Settings

In [ ]:
# ---- Change these, then run the cells below in order -----------------------------
DRIVE_DIR = "/content/drive/MyDrive/skynode"      # your folder on Google Drive
CLASSES = ["drone", "airplane", "bird"]            # what the model will learn, in this order

FRAME_EVERY_S = 1.0          # keep one frame per this many seconds of video
MAX_FRAMES_PER_VIDEO = 120   # so one long clip can't swamp the rest
VAL_FRACTION = 0.2           # share of footage held back to grade the model honestly

# Auto-labeling: a ready-made model draws first-guess boxes; YOU check them in step 4.
COCO_CLASSES = {"airplane": "airplane", "bird": "bird"}   # the ready-made model's word -> your class
WORLD_PROMPTS = {"drone": "drone"}                         # YOLO-World (finds things by NAME): word -> your class
COCO_CONF = 0.25
WORLD_CONF = 0.10            # lower on purpose: small, far-away drones score low

# Training
BASE_MODEL = "yolov8n.pt"    # small and fast; matches what brain/detector.py expects
EPOCHS = 60
IMGSZ = 640
RUN_NAME = "skynode"
RESUME = True                # carry on from the last saved checkpoint if there is one

# Optional: add an older labeled dataset (YOLO format) that you already uploaded to Drive.
EXISTING_DATASET = ""        # e.g. "/content/drive/MyDrive/skynode/old_dataset"  (leave "" to skip)

# Where the code lives
REPO_URL = "https://github.com/bsalsa2/Skynode"
REPO_BRANCH = "main"

## 2 · Set up (installs, GPU check, connect Drive)

In [ ]:
!pip install -q ultralytics

import datetime, json, shutil, sys
from pathlib import Path

import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path(DRIVE_DIR)
VIDEOS, RUNS, MODELS = DRIVE / "videos", DRIVE / "runs", DRIVE / "models"
for folder in (VIDEOS, RUNS, MODELS):
    folder.mkdir(parents=True, exist_ok=True)

if not Path("/content/Skynode").exists():
    !git clone -q --depth 1 -b {REPO_BRANCH} {REPO_URL} /content/Skynode
else:
    !git -C /content/Skynode pull -q
sys.path.insert(0, "/content/Skynode/training")
import skynode_data as sd

for table in (COCO_CLASSES, WORLD_PROMPTS):
    for ours in table.values():
        assert ours in CLASSES, f"{ours!r} is in a mapping but not in CLASSES"
print("Ready. Videos folder:", VIDEOS)

## 3 · Pull frames out of your videos and auto-label them
Frames are saved on Colab's temporary disk (not your Drive). The first-guess boxes come from two ready-made models:
a standard one for planes and birds, and YOLO-World for drones (it finds things by name, so it can miss tiny drones).
**These are guesses. Step 4 is where you fix them.**

In [ ]:
videos = sd.list_videos(VIDEOS)
assert videos, f"No videos found in {VIDEOS}. Upload some, wait for Drive to finish syncing, run again."

FRAMES = Path("/content/frames")
shutil.rmtree(FRAMES, ignore_errors=True)
frames, hint_of = [], {}
for video in videos:
    got = sd.extract_frames(video, FRAMES, every_s=FRAME_EVERY_S, max_frames=MAX_FRAMES_PER_VIDEO)
    hint = sd.class_hint(video.name, CLASSES)
    for path in got:
        hint_of[path] = hint
    frames += got
    print(f"{video.name}: {len(got)} frames" + (f"  (only '{hint}' boxes kept)" if hint else ""))
print("Total frames:", len(frames))

In [ ]:
from ultralytics import YOLO, YOLOWorld

labels = {path: [] for path in frames}

def prelabel(model, wanted, conf):
    """Run `model` on frames and store boxes for the classes in `wanted` (model's word -> ours)."""
    todo = [p for p in frames if hint_of[p] is None or hint_of[p] in wanted.values()]
    for start in range(0, len(todo), 32):
        batch = todo[start:start + 32]
        results = model.predict([str(p) for p in batch], conf=conf, verbose=False)
        for path, result in zip(batch, results):
            height, width = result.orig_shape
            for box, class_index in zip(result.boxes.xyxy.tolist(), result.boxes.cls.tolist()):
                ours = wanted.get(result.names[int(class_index)])
                if ours is None or hint_of[path] not in (None, ours):
                    continue
                line = sd.format_label(CLASSES.index(ours), *box, width, height)
                if line:
                    labels[path].append(line)

if COCO_CLASSES:
    prelabel(YOLO("yolov8n.pt"), COCO_CLASSES, COCO_CONF)
if WORLD_PROMPTS:
    world = YOLOWorld("yolov8s-worldv2.pt")
    world.set_classes(list(WORLD_PROMPTS))
    prelabel(world, WORLD_PROMPTS, WORLD_CONF)

counts = {name: 0 for name in CLASSES}
for lines in labels.values():
    for class_id, *_ in sd.parse_label_lines(lines):
        counts[CLASSES[class_id]] += 1
print("Auto-labeled boxes:", counts)
print("Frames with no box:", sum(1 for lines in labels.values() if not lines), "of", len(frames))

## 4 · Check the labels (this matters more than anything else)
The model learns exactly what the boxes say. Wrong boxes in = a worse model out.
Each picture has a number like `#12`. Look through the sheets, then in the next cell:
- **`DROP`** = throw the whole frame away (blurry, confusing, or you aren't sure).
- **`WIPE`** = keep the frame but delete its boxes (the box is on a cloud, or the sky really is empty).
Boxes that are *missing* on a real plane or drone are the most harmful mistake: drop those frames.

In [ ]:
from IPython.display import display
from PIL import Image

PER_SHEET = 48
PAGES_TO_SHOW = 6            # raise this to look at more (None = all)
pages = (len(frames) + PER_SHEET - 1) // PER_SHEET

def show_page(page):
    indexes = range(page * PER_SHEET, min(len(frames), (page + 1) * PER_SHEET))
    previews = [sd.draw_preview(frames[i], labels[frames[i]], CLASSES, caption=f"#{i}") for i in indexes]
    sheet = sd.make_contact_sheet(previews, columns=6)
    display(Image.fromarray(sheet[:, :, ::-1]))           # OpenCV is BGR; the viewer wants RGB

shown = pages if PAGES_TO_SHOW is None else min(pages, PAGES_TO_SHOW)
for page in range(shown):
    print(f"Sheet {page + 1} of {pages}")
    show_page(page)
if shown < pages:
    print(f"NOT SHOWN: sheets {shown + 1} to {pages} ({len(frames) - shown * PER_SHEET} frames). "
          "Their auto-labels will be used unchecked unless you raise PAGES_TO_SHOW and look.")

In [ ]:
DROP = []        # frame numbers to throw away, e.g. [3, 17, 18]
WIPE = []        # frame numbers whose boxes should be deleted, e.g. [5]

bad = [i for i in DROP + WIPE if not 0 <= i < len(frames)]
assert not bad, f"These numbers aren't frames: {bad}"
dropped = {frames[i] for i in DROP}
for i in WIPE:
    labels[frames[i]] = []
kept = [p for p in frames if p not in dropped]
print(f"Keeping {len(kept)} frames (dropped {len(dropped)}, wiped boxes on {len(set(WIPE) - set(DROP))}).")

## 5 · Build the dataset
Grading uses footage the model never trained on: whole videos are held back when you have two or more
(frames from the same video look almost identical, so mixing them would give a fake-good score).

In [ ]:
import yaml

names = sorted(p.name for p in kept)
train_names, val_names = sd.split_frames(names, val_fraction=VAL_FRACTION)
by_name = {p.name: p for p in kept}

DATASET = Path("/content/dataset")
builder = sd.DatasetBuilder(DATASET, CLASSES)
for split, group in (("train", train_names), ("val", val_names)):
    for name in group:
        builder.add(split, by_name[name], labels[by_name[name]])

if EXISTING_DATASET:
    old = Path(EXISTING_DATASET)
    info = old / "data.yaml"
    assert info.exists(), f"No data.yaml in {old}. Put the dataset's data.yaml there so class names can be matched."
    old_names = yaml.safe_load(info.read_text())["names"]
    old_names = [old_names[i] for i in sorted(old_names)] if isinstance(old_names, dict) else list(old_names)
    added, dropped_boxes = builder.add_existing(old, old_names)
    print(f"Merged {added} images from {old.name}; ignored {dropped_boxes} boxes of classes you don't use.")

DATA_YAML = builder.write_yaml()
summary = builder.summary()
for split in ("train", "val"):
    s = summary[split]
    print(f"{split:5} {s['images']:6} images ({s['empty']} empty sky)  boxes: {s['instances']}")
for warning in sd.dataset_warnings(summary):
    print("WARNING:", warning)

## 6 · Train
The first epochs are slow because the GPU is warming up. Checkpoints are saved to your Drive after every epoch.

In [ ]:
last = RUNS / RUN_NAME / "weights" / "last.pt"
if RESUME and last.exists():
    print("Resuming from", last)
    YOLO(str(last)).train(resume=True)
else:
    YOLO(BASE_MODEL).train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=16, device=0, workers=2, patience=20,
        project=str(RUNS), name=RUN_NAME, exist_ok=True, seed=0,
    )
best = RUNS / RUN_NAME / "weights" / "best.pt"
print("Best weights:", best, "exists:", best.exists())

## 7 · How good is it?
These scores come from your held-out footage only. They tell you how it does on **your kind of sky**, not on every sky.
`mAP50` of 0.5 is a rough start, 0.8+ is good. A class with very few validation boxes has a shaky score.

In [ ]:
metrics = YOLO(str(best)).val(data=str(DATA_YAML), split="val", verbose=False)
print(f"Overall  mAP50 = {metrics.box.map50:.3f}   mAP50-95 = {metrics.box.map:.3f}")
for index, class_id in enumerate(metrics.box.ap_class_index):
    print(f"  {CLASSES[int(class_id)]:10} mAP50 = {metrics.box.ap50[index]:.3f}")
missing = [CLASSES[i] for i in range(len(CLASSES)) if i not in set(int(c) for c in metrics.box.ap_class_index)]
if missing:
    print("Not scored (no validation boxes):", missing)

In [ ]:
# A few validation pictures with the model's own boxes, to look at with your own eyes.
import random
import cv2
sample = random.Random(1).sample(val_names, min(12, len(val_names)))
results = YOLO(str(best)).predict([str(by_name[n]) for n in sample], conf=0.25, verbose=False)
tiles = [r.plot(labels=True, conf=True) for r in results]
tiles = [cv2.resize(t, (360, 270)) for t in tiles]
display(Image.fromarray(sd.make_contact_sheet(tiles, columns=4)[:, :, ::-1]))

## 8 · Export for the brain
This makes an `.onnx` file (what `brain/detector.py` runs), saves it to **Drive ▸ skynode ▸ models**, and downloads it.
Copy it into `brain/models/` and update `config.toml` with the lines printed below.

In [ ]:
onnx_path = Path(YOLO(str(best)).export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True))
stamp = datetime.date.today().isoformat()
final = MODELS / f"skynode-{stamp}.onnx"
shutil.copyfile(onnx_path, final)
shutil.copyfile(best, MODELS / f"skynode-{stamp}.pt")          # keep the weights too, to train on later
print("Saved to Drive:", final)

print("\nPut this in brain/config.toml:\n")
print("[model]")
print(f'path = "models/{final.name}"')
print("target_classes =", json.dumps([c for c in CLASSES if c != "bird"]), "  # add \"bird\" if you want to track birds")

from google.colab import files
files.download(str(final))

### Next
- Put the `.onnx` in `brain/models/`, edit `config.toml` as printed above, and run `python -m brain.run --source some_video.mp4`.
- Test on a video you did **not** train on. That is the real test.
- Each new batch of videos makes the next round better: add them to `skynode/videos`, re-run, and compare the mAP50 numbers.